<a href="https://colab.research.google.com/github/Mariam-Jae/Individual-project/blob/main/AI_bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset

dataset = load_dataset("LabHC/bias_in_bios")
train_data = dataset["train"]
test_data = dataset["test"]

print(train_data[0])

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text

texts = [clean_text(x["hard_text"]) for x in train_data]
labels = [x["profession"] for x in train_data]
genders = [x["gender"] for x in train_data]

gender_swap = {
    " he ": " she ",
    " she ": " he ",
    " his ": " her ",
    " her ": " his ",
    " him ": " her ",
    " himself ": " herself ",
    " herself ": " himself "
}

gender_words = [
    "he","she","him","her","his","hers",
    "himself","herself","man","woman",
    "male","female"
]

def remove_gender_words(text):
    words = text.split()
    words = [w for w in words if w not in gender_words]
    return " ".join(words)

def swap_gender(text):
    for k, v in gender_swap.items():
        text = text.replace(k, v)
    return text
swapped_texts = [swap_gender(t) for t in texts]
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_original = vectorizer.fit_transform(texts)
X_swapped = vectorizer.transform(swapped_texts)
from sklearn.linear_model import LogisticRegression

model_biased = LogisticRegression(max_iter=1000)
model_biased.fit(X_original, labels)

# Train debiased model
model_debiased = LogisticRegression(max_iter=1000)
model_debiased.fit(X_swapped, labels)

from sklearn.metrics import accuracy_score

test_texts = [clean_text(x["hard_text"]) for x in test_data]
test_labels = [x["profession"] for x in test_data]

X_test = vectorizer.transform(test_texts)

pred_biased = model_biased.predict(X_test)
pred_debiased = model_debiased.predict(X_test)

print("Biased accuracy:", accuracy_score(test_labels, pred_biased))
print("Debiased accuracy:", accuracy_score(test_labels, pred_debiased))

import joblib

joblib.dump(model_biased, "biased_model.pkl")
joblib.dump(model_debiased, "debiased_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

import pandas as pd

# Convert HuggingFace dataset to pandas dataframe
test_df = pd.DataFrame(test_data)

# Select random CVs for participants
test_resumes = test_df.sample(50, random_state=42)


test_resumes = test_resumes.rename(columns={"hard_text": "resume_text"})

# Save CSV
test_resumes[['resume_text']].to_excel(
    "Resumes.xlsx",
    index=False
)


# Download file
from google.colab import files
files.download("Resumes.xlsx")
print(test_resumes.columns)






{'hard_text': 'He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates', 'profession': 21, 'gender': 0}
Biased accuracy: 0.8045907397874209
Debiased accuracy: 0.8019360243870434


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Index(['resume_text', 'profession', 'gender'], dtype='object')


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CV.xlsx to CV.xlsx


In [ ]:
import pandas as pd

df = pd.read_excel("CV.xlsx")

df.head()

,"{""input"": ""worked on data"", ""output"": ""Performed data cleaning, preprocessing, and visualization using Python libraries.""}"
0,"{""input"": ""worked on data"", ""output"": ""Perform..."
1,"{""input"": ""handled customers"", ""output"": ""Mana..."
2,"{""input"": ""internship in IT"", ""output"": ""Compl..."
3,"{""input"": ""did sales"", ""output"": ""Led B2B sale..."
4,"{""input"": ""good at teamwork"", ""output"": ""Colla..."


In [ ]:
print(df.columns)

Index(['{"input": "worked on data", "output": "Performed data cleaning, preprocessing, and visualization using Python libraries."}'], dtype='object')


In [ ]:
df = df.rename(columns={
    "output": "resume_text",
    "input": "label"
})

In [ ]:
print(df.columns)

Index(['{"input": "worked on data", "output": "Performed data cleaning, preprocessing, and visualization using Python libraries."}'], dtype='object')


In [ ]:
import pandas as pd

df = pd.read_excel("CV.xlsx")

In [ ]:
text_column = df.columns[0]

In [ ]:
import json

parsed_rows = df[text_column].apply(json.loads)

df = pd.json_normalize(parsed_rows)

In [ ]:
df.head()

,input,output
0,worked on data,"Performed data cleaning, preprocessing, and vi..."
1,handled customers,Managed customer inquiries and resolved issues...
2,internship in IT,Completed a 3-month IT internship focusing on ...
3,did sales,"Led B2B sales efforts, increasing client acqui..."
4,good at teamwork,Collaborated effectively with cross-functional...


In [ ]:
df = df.rename(columns={
    "output": "resume_text",
    "input": "label"
})

In [ ]:
df.head()

,label,resume_text
0,worked on data,"Performed data cleaning, preprocessing, and vi..."
1,handled customers,Managed customer inquiries and resolved issues...
2,internship in IT,Completed a 3-month IT internship focusing on ...
3,did sales,"Led B2B sales efforts, increasing client acqui..."
4,good at teamwork,Collaborated effectively with cross-functional...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(df["resume_text"])
y = df["label"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

LogisticRegression(max_iter=1000)

In [ ]:
import pickle

pickle.dump(model, open("dataset2_model.pkl","wb"))
pickle.dump(vectorizer, open("dataset2_vectorizer.pkl","wb"))

In [ ]:
def gender_swap(text):
    swaps = {
        " he ": " she ",
        " she ": " he ",
        " him ": " her ",
        " her ": " him ",
        " his ": " her ",
        " hers ": " his "
    }

    text = " " + text.lower() + " "

    for k, v in swaps.items():
        text = text.replace(k, v)

    return text.strip()

In [ ]:
df["resume_text_swapped"] = df["resume_text"].apply(gender_swap)

In [ ]:
df_original = df[["resume_text", "label"]]
df_swapped = df[["resume_text_swapped", "label"]]

df_swapped = df_swapped.rename(columns={"resume_text_swapped": "resume_text"})

df_combined = pd.concat([df_original, df_swapped])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(df_combined["resume_text"])
y = df_combined["label"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

LogisticRegression(max_iter=1000)

In [ ]:
import pickle

pickle.dump(model, open("dataset2_model.pkl","wb"))
pickle.dump(vectorizer, open("dataset2_vectorizer.pkl","wb"))

In [ ]:
from datasets import load_dataset
import numpy as np

dataset = load_dataset("LabHC/bias_in_bios")
train_data = dataset["train"]

# Extract profession labels
labels = [x["profession"] for x in train_data]

# Get unique IDs
unique_labels = np.unique(labels)
print("Unique profession IDs:", unique_labels)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-0ab65b32c47407(…):   0%|          | 0.00/64.9M [00:00<?, ?B/s]

data/test-00000-of-00001-5598c840ce8de1e(…):   0%|          | 0.00/24.9M [00:00<?, ?B/s]

data/dev-00000-of-00001-e6551072fff26949(…):   0%|          | 0.00/9.95M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/257478 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/99069 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/39642 [00:00<?, ? examples/s]

Unique profession IDs: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27]


In [ ]:
dataset = load_dataset("LabHC/bias_in_bios")

print(dataset["train"].features["profession"])


Value('int64')


In [ ]:
from datasets import load_dataset
import re
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Load Bias in Bios dataset

dataset = load_dataset("LabHC/bias_in_bios")

train_data = dataset["train"]
test_data = dataset["test"]

print(train_data[0])

# Text Cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text


# Remove Gender Words (Debiasing)

gender_words = [
    "he","she","him","her","his","hers",
    "himself","herself","man","woman",
    "male","female"
]

def remove_gender_words(text):
    words = text.split()
    words = [w for w in words if w not in gender_words]
    return " ".join(words)



# Prepare Training Data

texts = [clean_text(x["hard_text"]) for x in train_data]
labels = [x["profession"] for x in train_data]

# Create debiased version
debiased_texts = [remove_gender_words(t) for t in texts]


# Vectorization

vectorizer = TfidfVectorizer(max_features=15000)

X_original = vectorizer.fit_transform(texts)
X_debiased = vectorizer.transform(debiased_texts)


# Train Models

model_biased = LogisticRegression(max_iter=1000)
model_biased.fit(X_original, labels)

model_debiased = LogisticRegression(max_iter=1000)
model_debiased.fit(X_debiased, labels)


# Evaluate Models

test_texts = [clean_text(x["hard_text"]) for x in test_data]
test_labels = [x["profession"] for x in test_data]

X_test = vectorizer.transform(test_texts)

pred_biased = model_biased.predict(X_test)
pred_debiased = model_debiased.predict(X_test)

print("Biased accuracy:", accuracy_score(test_labels, pred_biased))
print("Debiased accuracy:", accuracy_score(test_labels, pred_debiased))


# Save Models

joblib.dump(model_biased, "biased_model.pkl")
joblib.dump(model_debiased, "debiased_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("Models saved!")



# Create CV Dataset for Streamlit

test_df = pd.DataFrame(test_data)

# Select random CVs for the demo
test_resumes = test_df.sample(50, random_state=42)

# Rename column for Streamlit
test_resumes = test_resumes.rename(columns={"hard_text": "resume_text"})

# Save to Excel
test_resumes[['resume_text']].to_excel(
    "Resumes.xlsx",
    index=False
)

print("Resume dataset saved!")



# Download Files

from google.colab import files

files.download("biased_model.pkl")
files.download("debiased_model.pkl")
files.download("vectorizer.pkl")
files.download("Resumes.xlsx")

{'hard_text': 'He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates', 'profession': 21, 'gender': 0}
Biased accuracy: 0.8099607344376142
Debiased accuracy: 0.8085778598754404
Models saved!
Resume dataset saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from datasets import load_dataset
import re
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Load dataset

dataset = load_dataset("LabHC/bias_in_bios")

train_data = dataset["train"]
test_data = dataset["test"]

print(train_data[0])



# Text cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text



# Gender removal (for debiased model)

gender_words = [
    "he","she","him","her","his","hers",
    "himself","herself","man","woman",
    "male","female"
]

def remove_gender_words(text):
    words = text.split()
    words = [w for w in words if w not in gender_words]
    return " ".join(words)



# Bias amplification

def amplify_gender(text):
    text = text.replace(" he ", " he he he ")
    text = text.replace(" she ", " she she she ")
    text = text.replace(" his ", " his his his ")
    text = text.replace(" her ", " her her her ")
    return text



# Prepare training data

texts = [clean_text(x["hard_text"]) for x in train_data]
labels = [x["profession"] for x in train_data]

# amplify bias for biased model
biased_texts = [amplify_gender(t) for t in texts]

# remove gender for debiased model
debiased_texts = [remove_gender_words(t) for t in texts]


# Vectorization

vectorizer = TfidfVectorizer(max_features=15000)

X_biased = vectorizer.fit_transform(biased_texts)
X_debiased = vectorizer.transform(debiased_texts)



# Train models
model_biased = LogisticRegression(max_iter=1000)
model_biased.fit(X_biased, labels)

model_debiased = LogisticRegression(max_iter=1000)
model_debiased.fit(X_debiased, labels)



# Evaluate models
test_texts = [clean_text(x["hard_text"]) for x in test_data]
test_labels = [x["profession"] for x in test_data]

X_test = vectorizer.transform(test_texts)

pred_biased = model_biased.predict(X_test)
pred_debiased = model_debiased.predict(X_test)

print("Biased accuracy:", accuracy_score(test_labels, pred_biased))
print("Debiased accuracy:", accuracy_score(test_labels, pred_debiased))



# Save models
joblib.dump(model_biased, "biased_model.pkl")
joblib.dump(model_debiased, "debiased_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("Models saved!")



# Create CV dataset for app
test_df = pd.DataFrame(test_data)

test_resumes = test_df.sample(50, random_state=42)

test_resumes = test_resumes.rename(columns={"hard_text": "resume_text"})

test_resumes[['resume_text']].to_excel(
    "Resumes.xlsx",
    index=False
)

print("Resume dataset saved!")



# Download files


from google.colab import files

files.download("biased_model.pkl")
files.download("debiased_model.pkl")
files.download("vectorizer.pkl")
files.download("Resumes.xlsx")

{'hard_text': 'He is also the project lead of and major contributor to the open source assembler/simulator "EASy68K." He earned a master’s degree in computer science from the University of Michigan-Dearborn, where he is also an adjunct instructor. Downloads/Updates', 'profession': 21, 'gender': 0}
Biased accuracy: 0.8092440622192614
Debiased accuracy: 0.8085778598754404
Models saved!
Resume dataset saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>